In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import time

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print(f"Loading {MODEL_NAME}...")
start = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print(f"Loaded in {time.time() - start:.1f}s")

prompt = """
Improve this resume bullet for a Junior ML Engineer role:

Built machine learning models using Python.

Be specific, add metrics if possible, and use strong action verbs.
"""

messages = [{"role": "user", "content": prompt}]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[1]

for temp in [0.2, 0.7, 1.2]:
    print("=" * 50)
    print(f"Temperature: {temp}")
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            temperature=temp,
            top_k=50,
            top_p=0.9,
            do_sample=True,
            max_new_tokens=200,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][input_len:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True)
    print(result)
    print(f"({time.time() - t0:.1f}s)\n")

Loading HuggingFaceTB/SmolLM2-1.7B-Instruct...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Loaded in 12.4s
Temperature: 0.2
Here's a revised version of the bullet point:

"Designed and implemented a range of machine learning models using Python, achieving a 95% accuracy rate in predictive modeling and a 20% reduction in processing time. Utilized advanced algorithms to optimize model performance and developed a custom framework for efficient data processing, resulting in a 30% improvement in model efficiency."
(2.3s)

Temperature: 0.7
Here's a revised version of the resume bullet with more specificity, added metrics, and stronger action verbs:

"Designed and implemented scalable machine learning models using Python to achieve a 25% reduction in prediction error and an average model accuracy of 85% across 500+ real-world datasets."
(1.8s)

Temperature: 1.2
Sure, here's a revised bullet for a Junior ML Engineer role:

Consistently developed and validated machine learning models using high-performance computing. Collected and processed large datasets to identify patterns, genera

In [14]:
# Cell 2 — Full analyzer pipeline (same logic as analyzer.py, readable inline)

# ─── Helper: similarity (from similarity.py)
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

SIMILARITY_THRESHOLD = 0.50

def get_best_matches(resume_sentences, jd_sentences):
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    resume_embeddings = embedding_model.encode(resume_sentences)
    jd_embeddings = embedding_model.encode(jd_sentences)
    sim_matrix = cosine_similarity(jd_embeddings, resume_embeddings)

    matches = []
    for i, jd in enumerate(jd_sentences):
        best_index = sim_matrix[i].argmax()
        matches.append({
            "jd": jd,
            "resume": resume_sentences[best_index],
            "score": float(sim_matrix[i][best_index])
        })
    return matches

# ─── Helper: generate_text wraps model from Cell 1 (from llm.py)
import torch

def generate_text(prompt, temperature=0.7, top_k=50, top_p=0.9, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs, temperature=temperature, top_k=top_k, top_p=top_p,
            do_sample=True, max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

# ─── Data (same as data/resume_sentences.txt and data/jd_sentences.txt)
resume_sentences = [
    "Built MLflow pipeline for experiment tracking.",
    "Implemented DVC for dataset versioning.",
    "Developed CNN models using PyTorch.",
    "Created FastAPI applications for deployment.",
    "Built customer segmentation machine learning pipeline.",
]

jd_sentences = [
    "Experience with MLOps workflows.",
    "Knowledge of Docker and Kubernetes.",
    "Experience building NLP applications.",
    "Strong Python programming skills.",
    "Experience deploying machine learning models.",
]

# ─── === analyzer.py logic starts here ===
matches = get_best_matches(resume_sentences, jd_sentences)
print("=" * 50)
print("TOP RESUME MATCHES")
print("=" * 50)

filtered_matches = [m for m in matches if m['score'] >= SIMILARITY_THRESHOLD]
for match in filtered_matches:
    print(f"\nJD: {match['jd']}")
    print(f"Best Resume Match: {match['resume']}")
    print(f"Score: {match['score'] * 100:.1f}%")

# --- LLM gap analysis (the Day 1 addition) ---
low_matches = [m for m in matches if m['score'] < SIMILARITY_THRESHOLD]
gap_skills = [m['jd'] for m in low_matches]
suggestion = generate_text(
    f"Suggest how to improve a resume for someone missing these skills: {', '.join(gap_skills[:3])}"
)

print("\n" + "=" * 50)
print("LLM SUGGESTION")
print("=" * 50)
print(suggestion)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

TOP RESUME MATCHES

LLM SUGGESTION
To improve a resume for someone missing these skills, consider the following suggestions:

1. **Add relevant work experience**: Even if you don't have direct experience with MLOps workflows, you may have handled similar tasks in previous roles. Highlight any projects or tasks that involved managing or deploying machine learning models.

2. **Include relevant courses or certifications**: If you've taken courses or earned certifications in MLOps, Docker, or NLP, be sure to include them in your resume. This demonstrates your commitment to learning and your willingness to take on new challenges.

3. **Highlight transferable skills**: Even if you don't have direct experience with these skills, you may have skills that are transferable. For example, if you have experience with data pipelines, version control systems (like Git), or data visualization tools, highlight these skills.

4. **Include relevant projects or contributions**: If you've worked on projec

In [21]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

SIMILARITY_THRESHOLD = 0.65

def compute_similarity(resume_sentences, jd_sentences):

    embedding_model  = SentenceTransformer("all-MiniLM-L6-v2")

    resume_embeddings = embedding_model.encode(resume_sentences)
    jd_embeddings = embedding_model.encode(jd_sentences)

    similarity_matrix = cosine_similarity(jd_embeddings, resume_embeddings)

    return similarity_matrix

def get_best_matches(resume_sentences, jd_sentences):

    matches = []

    similarity_matrix = compute_similarity(
        resume_sentences,
        jd_sentences
    )

    for i, jd in enumerate(jd_sentences):

        best_index = similarity_matrix[i].argmax()

        best_score = similarity_matrix[i][best_index]

        best_resume = resume_sentences[best_index]

        matches.append({
            "jd": jd,
            "resume": best_resume,
            "score": float(best_score)
        })

    return matches



In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

QA_MODEL_NAME = "distilbert-base-cased-distilled-squad"

qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)

def answer_question(context, question):
    inputs = qa_tokenizer(question, context, return_tensors="pt", truncation=True, max_length=512)
    output = qa_model(**inputs)
    start_idx = torch.argmax(output.start_logits)
    end_idx = torch.argmax(output.end_logits)
    if start_idx == 0 and end_idx == 0:
        return "No specific answer found \u2014 try rephrasing the question"
    answer_ids = inputs["input_ids"][0, start_idx:end_idx+1]
    answer = qa_tokenizer.decode(answer_ids, skip_special_tokens=True)
    return answer if answer.strip() else "No specific answer found \u2014 try rephrasing the question"


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SUM_MODEL_NAME = "facebook/bart-large-cnn"

sum_tokenizer = AutoTokenizer.from_pretrained(SUM_MODEL_NAME)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(SUM_MODEL_NAME)

def summarize_jd(jd_text):
    inputs = sum_tokenizer(jd_text, return_tensors="pt", truncation=True, max_length=1024)
    summary_ids = sum_model.generate(
        inputs["input_ids"],
        num_beams=4, min_length=20, max_length=80, do_sample=False
    )
    return sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

In [24]:
real_jd = """
Job Title: Junior Data Scientist / Forecasting Analyst

Experience Required:
3–4 Years Overall Experience

Relevant Experience:
Minimum 1.5+ Years in Data Science, Machine Learning, or Predictive Analytics

About the Role

We are seeking a motivated and detail-oriented Junior Data Scientist / Forecasting Analyst to support our Data Science team in the development, execution, and maintenance of predictive analytics solutions focused on demand forecasting.

The ideal candidate will work closely with senior data scientists and business stakeholders, contributing to data preparation, model execution, forecasting operations, and continuous process improvement.

Key Responsibilities

Support the implementation, execution, and maintenance of demand forecasting and predictive analytics solutions.

Assist in analyzing historical sales data, demand patterns, and business variables to generate actionable insights.

Execute existing forecasting models under the guidance of senior team members.

Perform model retraining activities and generate periodic forecasts according to established schedules.

Monitor model outputs and ensure timely delivery of forecasting results.

Prepare, clean, and validate datasets used for predictive modeling.

Perform data quality assessments and identify anomalies, inconsistencies, and missing data.

Support feature engineering activities and improve data readiness for model inputs.

Calculate and report forecast accuracy metrics and model performance indicators.

Monitor forecasting outputs and identify deviations, anomalies, or unexpected trends.

Assist in automating data preparation, model execution, and prediction generation processes.

Develop and maintain Python scripts and simple data pipelines.

Support workflow optimization initiatives to improve efficiency and scalability.

Prepare reports, dashboards, visualizations, and result summaries.

Maintain technical documentation for models, datasets, processes, and workflows.

Required Qualifications

Bachelor's or Master's degree in Data Science, Computer Science, Statistics, Mathematics, Engineering, Economics, or a related quantitative discipline.

Minimum 1.5+ years of hands-on experience in Data Science, Machine Learning, Predictive Analytics, or Forecasting-related projects.

Experience working with structured datasets and analytical workflows.

Mandatory Technical Skills

Strong proficiency in Python.

Pandas.

NumPy.

Scikit-learn.

Experience with Git version control.

Strong understanding of SQL fundamentals.

Understanding of machine learning fundamentals.

Knowledge of regression techniques.

Model validation methodologies.

Forecast accuracy measurements.

Statistical analysis concepts.

Error metrics (MAE, RMSE, MAPE).

Data cleaning and preprocessing.

Data quality validation and anomaly detection.

Exploratory Data Analysis (EDA).

Preferred Skills

Demand Forecasting methodologies.

Time Series Analysis and Forecasting.

PySpark.

Azure Databricks.

Power BI.

Tableau.

Matplotlib.

Seaborn.

Cloud-based analytics platforms.

Basic understanding of MLOps concepts and model lifecycle management.
"""

In [25]:
real_resume = """
Data Scientist | ML Engineer

Professional Summary

Data Science professional with 10+ months of hands-on experience building end-to-end machine learning pipelines and MLOps systems.

Proficient in Python, Scikit-learn, PyTorch, Docker, FastAPI, GitHub Actions, DVC, and MLflow.

Experienced in computer vision, classical machine learning, deep learning, NLP, and production-grade REST API deployment.

Technical Skills

Python.

SQL (PostgreSQL).

Scikit-learn.

PyTorch.

Machine Learning.

Deep Learning.

Computer Vision.

Natural Language Processing.

Docker.

FastAPI.

Flask.

Git.

GitHub Actions.

DVC.

MLflow.

Power BI.

Pandas.

NumPy.

Matplotlib.

Seaborn.

EDA.

Feature Engineering.

Statistics.

Hypothesis Testing.

Data Wrangling.

Professional Experience

Data Science Intern at Bridgeon Solutions.

Completed a structured 26-week intensive bootcamp covering statistics, SQL, Python, machine learning, deep learning, computer vision, and MLOps.

Designed and deployed an end-to-end MLOps pipeline using DVC, MLflow, Docker, FastAPI, and GitHub Actions.

Implemented automated CI/CD workflows and weekly data drift monitoring using EvidentlyAI.

Trained and evaluated more than 15 machine learning models across supervised, unsupervised, and deep learning paradigms.

Projects

Customer Segmentation MLOps Pipeline.

Built customer segmentation solution using KMeans clustering and PCA.

Engineered customer segments from 2,240 records and deployed a complete MLOps stack using DVC, MLflow, FastAPI, Docker, GitHub Actions, and EvidentlyAI.

Waste Classification CNN.

Built a 6-class image classification model using PyTorch CNN.

Implemented MLflow experiment tracking, DVC versioning, Docker containerization, and GitHub Actions CI/CD.

Bank Marketing Subscription Prediction.

Built a machine learning pipeline using Random Forest and Scikit-learn.

Evaluated multiple machine learning models and selected the best-performing model based on ROC-AUC.

Developed Flask API, Streamlit dashboard, and Dockerized deployment.

Olist E-Commerce Sales Dashboard.

Developed a Power BI dashboard analyzing over 100,000 orders.

Created KPI reports, business insights, and executive-level recommendations.

Education

B.Sc. Electronics.

University of Calicut.

Graduated in 2025.

"""

In [26]:
# Split the full resume/JD text into non-empty lines (same effect as load_sentences)
resume_sentences = [line.strip() for line in real_resume.strip().split("\n") if line.strip()]
jd_sentences = [line.strip() for line in real_jd.strip().split("\n") if line.strip()]

def analyze(resume_sentences, jd_sentences):
    if not resume_sentences:
        raise ValueError("Resume is empty.")
    if not jd_sentences:
        raise ValueError("Job description is empty.")

    matches = get_best_matches(resume_sentences, jd_sentences)
    print("=" * 50)
    print("TOP RESUME MATCHES")
    print("=" * 50)

    filtered_matches = [match for match in matches if match['score'] >= SIMILARITY_THRESHOLD]
    for match in filtered_matches:
        print(f"\nJD: {match['jd']}")
        print(f"Best Resume Match: {match['resume']}")
        print(f"Score: {match['score'] * 100:.1f}%")

    jd_text = "\n".join(jd_sentences)

    summary = summarize_jd(jd_text)
    print("\n" + "=" * 50)
    print("JD SUMMARY")
    print("=" * 50)
    print(summary)

    questions = [
        "What is the job title?",
        "What experience is required?",
        "What tools are mentioned?",
        "What technical skills are mandatory?",
    ]
    for question in questions:
        answer = answer_question(jd_text, question)
        print("\n" + "=" * 50)
        print("QUESTION ANSWERING")
        print("=" * 50)
        print(f"Question: {question}")
        print(f"Answer: {answer}")

    low_matches = [match for match in matches if match['score'] < SIMILARITY_THRESHOLD]
    gap_skills = [m['jd'] for m in low_matches]
    suggestion = generate_text(
        f"The candidate's resume appears weak in these areas: {', '.join(gap_skills[:3])} Suggest 3 concrete improvements to strengthen the resume."
    )

    print("\n" + "=" * 50)
    print("LLM SUGGESTION")
    print("=" * 50)
    print(suggestion)

analyze(resume_sentences, jd_sentences)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

TOP RESUME MATCHES

JD: Experience Required:
Best Resume Match: Professional Experience
Score: 70.4%

JD: Mandatory Technical Skills
Best Resume Match: Technical Skills
Score: 79.0%

JD: Pandas.
Best Resume Match: Pandas.
Score: 100.0%

JD: NumPy.
Best Resume Match: NumPy.
Score: 100.0%

JD: Scikit-learn.
Best Resume Match: Scikit-learn.
Score: 100.0%

JD: Power BI.
Best Resume Match: Power BI.
Score: 100.0%

JD: Matplotlib.
Best Resume Match: Matplotlib.
Score: 100.0%

JD: Seaborn.
Best Resume Match: Seaborn.
Score: 100.0%

JD SUMMARY
Junior Data Scientist / Forecasting Analyst is required to have 1.5+ years of hands-on experience in Data Science, Machine Learning, Predictive Analytics, or Forecasting-related projects. The ideal candidate will work closely with senior data scientists and business stakeholders.

QUESTION ANSWERING
Question: What is the job title?
Answer: Junior Data Scientist

QUESTION ANSWERING
Question: What experience is required?
Answer: No specific answer found — 